# 📚 Tutorial Complet — IA Finance

Ce notebook explique **etape par etape** comment fonctionne le projet IA Finance.

## Sommaire
1. Collecte des donnees
2. Nettoyage et feature engineering
3. Modele Prophet
4. Modele XGBoost
5. Modele LSTM
6. Comparaison et predictions

---
## Etape 1 : Collecte des donnees

On utilise **yfinance** pour telecharger l'historique boursier depuis Yahoo Finance.

C'est gratuit et ne necessite aucune cle API.

In [ ]:
import yfinance as yf
import pandas as pd

# Telecharger 1 an de donnees pour Apple
data = yf.download('AAPL', period='1y', interval='1d', progress=False)

# Aplatir les colonnes (yfinance retourne un MultiIndex)
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

print(f'Donnees telechargees: {len(data)} lignes')
print(f'Colonnes: {data.columns.tolist()}')
data.head()

### Que signifient les colonnes ?

| Colonne | Signification |
|---------|---------------|
| **Open** | Prix d'ouverture (debut de journee) |
| **High** | Prix le plus haut de la journee |
| **Low** | Prix le plus bas de la journee |
| **Close** | Prix de cloture (fin de journee) — c'est ce qu'on predit |
| **Volume** | Nombre d'actions echangees |

---
## Etape 2 : Nettoyage et Feature Engineering

On nettoie les donnees et on ajoute des **indicateurs techniques** qui aident les modeles.

In [ ]:
# Rendement journalier : combien le prix a change en %
data['Return_1d'] = data['Close'].pct_change()

# Moyennes mobiles : tendance sur 7, 20, 50 jours
data['MA_7'] = data['Close'].rolling(7).mean()
data['MA_20'] = data['Close'].rolling(20).mean()
data['MA_50'] = data['Close'].rolling(50).mean()

# Volatilite : a quel point le prix bouge
data['Volatility_20d'] = data['Return_1d'].rolling(20).std()

print('Features ajoutees!')
data[['Close', 'Return_1d', 'MA_7', 'MA_20', 'Volatility_20d']].tail(10)

In [ ]:
# RSI (Relative Strength Index) — indicateur de surachat/survente
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

data['RSI_14'] = compute_rsi(data['Close'])

print('RSI > 70 = surachete (le prix va probablement baisser)')
print('RSI < 30 = survendu (le prix va probablement monter)')
print(f'\nRSI actuel: {data["RSI_14"].iloc[-1]:.2f}')

---
## Etape 3 : Modele Prophet

**Prophet** (par Meta/Facebook) decompose une serie temporelle en :
- **Tendance** : le prix monte ou descend globalement ?
- **Saisonnalite** : y a-t-il des patterns qui se repetent (semaine, annee) ?
- **Points de changement** : ou la tendance change de direction ?

In [ ]:
from prophet import Prophet
import plotly.graph_objects as go

# Prophet a besoin de 2 colonnes : ds (date) et y (valeur)
prophet_data = data[['Close']].reset_index()
prophet_data.columns = ['ds', 'y']

# Split train/test
split = int(len(prophet_data) * 0.8)
train = prophet_data[:split]
test = prophet_data[split:]

print(f'Train: {len(train)} jours')
print(f'Test: {len(test)} jours')

# Entrainement
model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
model.fit(train)

# Prevision
future = model.make_future_dataframe(periods=len(test))
forecast = model.predict(future)

print('Modele entraine!')

In [ ]:
# Visualisation
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test['ds'], y=test['y'], name='Test (reel)', line=dict(color='green')))
fig.add_trace(go.Scatter(x=forecast['ds'].tail(len(test)), y=forecast['yhat'].tail(len(test)),
                         name='Prediction Prophet', line=dict(color='red', dash='dash')))
fig.update_layout(title='Prophet : Prediction vs Realite', height=450)
fig.show()

# Metrique
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(test['y'], forecast['yhat'].tail(len(test)))
print(f'\nMAE Prophet: ${mae:.2f} (erreur moyenne en $)')

---
## Etape 4 : Modele XGBoost

**XGBoost** (Extreme Gradient Boosting) utilise les features techniques comme entrees pour predire le prix.

C'est comme si on donnait a un expert tous les indicateurs et il decide du prix.

In [ ]:
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import numpy as np

# Preparer les features
features = ['Open', 'High', 'Low', 'Volume', 'MA_7', 'MA_20', 'MA_50', 'Volatility_20d', 'RSI_14']
clean_data = data.dropna()

X = clean_data[features].values
y = clean_data['Close'].values

# Split temporel
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Normaliser
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Entrainer
model_xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, verbosity=0)
model_xgb.fit(X_train_s, y_train)

# Predire
y_pred = model_xgb.predict(X_test_s)

mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
print(f'MAE XGBoost: ${mae:.2f}')
print(f'MAPE XGBoost: {mape:.2f}%')

In [ ]:
# Visualisation predictions vs realite
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test, name='Reel', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=y_pred, name='Prediction XGBoost', line=dict(color='red', dash='dash')))
fig.update_layout(title='XGBoost : Prediction vs Realite', 
                  xaxis_title='Jours', yaxis_title='Prix ($)', height=400)
fig.show()

In [ ]:
# Quelles features sont les plus importantes ?
importance = pd.Series(model_xgb.feature_importances_, index=features).sort_values()
fig = px.barh(x=importance.values, y=importance.index,
              title='Importance des features (XGBoost)')
fig.show()

print('\n-> High et Low sont les features les plus importantes')
print('   (logique : le prix de cloture est entre le High et le Low)')

---
## Etape 5 : Modele LSTM (Deep Learning)

**LSTM** (Long Short-Term Memory) est un reseau de neurones qui :
- Regarde les **60 derniers jours** de prix
- Apprend les patterns sequentiels
- Predit le prix du jour suivant

C'est comme si on montrait un graphique a quelqu'un et on lui demandait "ou va aller la courbe?".

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

# Preparer les donnees
close_prices = data['Close'].dropna().values.reshape(-1, 1)

# Normaliser entre 0 et 1
scaler_lstm = MinMaxScaler()
data_scaled = scaler_lstm.fit_transform(close_prices)

# Creer des sequences de 60 jours
SEQ_LENGTH = 60
X_seq, y_seq = [], []
for i in range(len(data_scaled) - SEQ_LENGTH):
    X_seq.append(data_scaled[i:i+SEQ_LENGTH])
    y_seq.append(data_scaled[i+SEQ_LENGTH])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# Split
split = int(len(X_seq) * 0.8)
X_train_l = torch.FloatTensor(X_seq[:split])
y_train_l = torch.FloatTensor(y_seq[:split])
X_test_l = torch.FloatTensor(X_seq[split:])
y_test_l = torch.FloatTensor(y_seq[split:])

print(f'Sequences d\'entrainement: {len(X_train_l)}')
print(f'Sequences de test: {len(X_test_l)}')
print(f'Chaque sequence = {SEQ_LENGTH} jours de prix')

In [ ]:
# Definir le reseau LSTM
class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2, 
                           batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# Entrainer
model_lstm = SimpleLSTM()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

print('Entrainement LSTM...')
model_lstm.train()
for epoch in range(30):
    optimizer.zero_grad()
    output = model_lstm(X_train_l)
    loss = criterion(output, y_train_l)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'  Epoch {epoch+1}/30 - Loss: {loss.item():.6f}')

print('Entrainement termine!')

In [ ]:
# Evaluer
model_lstm.eval()
with torch.no_grad():
    pred_scaled = model_lstm(X_test_l).numpy()

# Denormaliser
y_test_real = scaler_lstm.inverse_transform(y_test_l.numpy())
y_pred_real = scaler_lstm.inverse_transform(pred_scaled)

mae_lstm = np.mean(np.abs(y_test_real - y_pred_real))
print(f'MAE LSTM: ${mae_lstm:.2f}')

# Visualisation
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test_real.flatten(), name='Reel', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=y_pred_real.flatten(), name='Prediction LSTM', 
                         line=dict(color='red', dash='dash')))
fig.update_layout(title='LSTM : Prediction vs Realite',
                  xaxis_title='Jours', yaxis_title='Prix ($)', height=400)
fig.show()

---
## Etape 6 : Comparaison finale

### Resume des resultats

| Modele | Approche | Quand l'utiliser |
|--------|----------|------------------|
| **Prophet** | Tendance + saisonnalite | Previsions long terme (30+ jours) |
| **XGBoost** | Features techniques | Prediction precise a court terme |
| **LSTM** | Sequences de prix | Capture des patterns complexes |

### Prediction Ensemble

En combinant les 3 modeles (moyenne), on obtient souvent un meilleur resultat
car chaque modele capture des aspects differents du marche.

In [ ]:
print('=== RESUME ===')
print(f'\nPour predire le prochain prix de AAPL:')
print(f'  Prophet pense : ${forecast["yhat"].iloc[-1]:.2f}')
print(f'  XGBoost pense : ${y_pred[-1]:.2f}')
print(f'  LSTM pense    : ${y_pred_real[-1][0]:.2f}')
ensemble = np.mean([forecast['yhat'].iloc[-1], y_pred[-1], y_pred_real[-1][0]])
print(f'\n  Ensemble (moyenne) : ${ensemble:.2f}')
print(f'\n  Prix reel : ${close_prices[-1][0]:.2f}')